# WASP-6: Exo-Striker from Python

**Example provided by Trifon Trifonov with the aid of CODEX; use with caution.**

Fit the bundled detrended relative-flux light curve with Exo-Striker's Batman
transit likelihood. The file contains 18,566 measurements over about 27 days and
multiple transits. RV, GP, TTV, and astrometry likelihoods are disabled.
The first three columns are BJD, detrended flux, and its uncertainty. The remaining
columns store detrending provenance; no airmass or extra baseline trend is fitted.

This tutorial follows the fitting → priors → nested sampling → diagnostics →
CSV exports structure of `Alans_NB/NS_nu_Oph_eng_v3.ipynb`, updated for the current
Exo-Striker API. It starts from the repository data files; no old session is needed.
The matching script is `WASP6_transit.py`. Shared plotting/export utilities live in
`example_tools.py`.

Run the cells in order. Nested sampling is optional and disabled initially, so
**Run All** first produces a best-fit analysis. Enable it explicitly below for
posterior uncertainties and evidence; a full run can take minutes to hours.

## 1. Setup and paths

Use a Python environment with this checkout's Exo-Striker dependencies installed
(`python -m pip install -e .` from the repository root). The first import may
compile the native RV backend, requiring a working Fortran compiler, Meson and Ninja.
No external LaTeX installation or downloads of observational data are needed.
Open Jupyter from the repository root or `examples/`; paths are resolved before
importing Exo-Striker, because older releases change the working directory.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / 'exostriker' / 'datafiles').is_dir()
             and (p / 'examples' / 'example_tools.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or examples directory.')
sys.path.insert(0, str(ROOT / 'examples'))
from example_tools import (DATA, EXAMPLES, rv, np, plt, uniform_bound,
                           check_priors, plot_transit, finish, command_line)
from example_tools import configure_sampling, export_posterior, save_session

In [ ]:
TARGET = 'WASP6'
OUTPUT = EXAMPLES / 'results' / TARGET
RUN_NESTED = False   # Set True to run the optional posterior analysis.
NS_THREADS = 1       # Start with one process; increase deliberately for long runs.
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Data:', DATA)
print('Outputs:', OUTPUT)

## 2. Data, starting model, and explicit priors

The functions below show every starting value and prior. Parameter slots are
zero-based. Uniform bounds are set explicitly and Gaussian/log-uniform priors
are disabled for those parameters. These are tutorial choices, not evidence for
a particular prior. For model comparisons, choose justified priors before the
analysis and keep shared priors consistent across models.

In [ ]:
def set_priors(fit):
    for key, limits in {'P': (3.35, 3.37), 't0': (2458353.85, 2458354.2), 'pl_a': (5., 20.),
                        'pl_rad': (.05, .25), 'i': (80., 90.)}.items():
        uniform_bound(fit, key, 0, limits)
    uniform_bound(fit, 'tra_off', 0, (-.02, .02), plural=True)
    uniform_bound(fit, 'tra_jitt', 0, (0., .01), plural=True)

In [ ]:
def build_fit():
    fit = rv.signal_fit(name='WASP-6 — transit-only example')
    fit.type_fit = {'RV': False, 'Transit': True, 'TTV': False, 'AST': False}
    fit.rtg = [False, False, True, False]  # RV, RV GP, transit, transit GP
    fit.init_fit = False
    fit.stellar_mass = fit.params.stellar_mass = .9  # illustrative; fixed
    fit.add_transit_dataset('WASP-6 detrended', str(DATA / 'WASP_6_lc_clean.tran'))
    # Only columns 1–3 are used for time/flux/error; the extra columns in the
    # clean file record its detrending provenance. No airmass term is fitted.
    fit.tra_data_sets[0][3] = np.zeros_like(fit.tra_data_sets[0][0])
    fit.tra_data_sets_init[0][3] = np.zeros_like(fit.tra_data_sets[0][0])
    fit.add_planet(K=0., P=3.361, e=0., w=90., M0=0., i=88.5,
                   useK=False, useP=True, usee=False, usew=False,
                   useM0=False, usei=True)
    # The detrended file spans 27 days and multiple transits. Fit the period,
    # conjunction time, a/Rstar, radius ratio and inclination together.
    fit.t0[0] = 2458354.03
    fit.epoch = fit.t0[0]
    fit.pl_a[0] = 11.
    fit.pl_rad[0] = .145
    fit.t0_use[0] = fit.pl_a_use[0] = fit.pl_rad_use[0] = True
    fit.tra_off[0] = 0.
    fit.tra_jitt[0] = .0005
    fit.tra_jitt_use[0] = False
    fit.ld_m[0] = 'quadratic'
    fit.ld_u[0] = fit.ld_u_quad[0] = [.12, .35]
    fit.ld_u_quad_use[0] = [False, False]
    fit.tra_lintr_use[0] = fit.tra_quadtr_use[0] = False
    fit.tra_model_fact = 10  # integer oversampling factor for NumPy linspace
    fit.SciPy_min_use_1 = 'Nelder-Mead'
    fit.SciPy_min_N_use_1 = 2
    fit.SciPy_min_N_use_2 = 0
    fit.Simplex_opt.update(disp=False, maxiter=10000, maxfev=20000, xatol=1e-7, fatol=1e-6)
    fit.Simplex_custom_steps = True
    set_priors(fit)
    return fit

fit = build_fit()
print("Reference epoch [JD]:", fit.epoch)
check_priors(fit)

## 3. Optimize the model

Fit period, mid-transit time, inclination, scaled semimajor axis `a/Rstar`,
radius ratio `Rp/Rstar`, and a flux offset. First optimize with fixed jitter,
then fit an additional white-noise term. Fix eccentricity to zero, omega to 90°,
and quadratic limb-darkening coefficients to `[0.12, 0.35]` (illustrative values).
The period starts at 3.361 days and is free over [3.35, 3.37] days. Stellar mass is
fixed at an illustrative 0.9 solar masses; `a/Rstar` is fitted independently
(`rho=False`). These assumptions and the supplied detrending affect the result;
this is an instructional analysis, not a new published parameter determination.

Covariance estimates from optimization are not a substitute for the posterior intervals below.

In [ ]:
def optimize(fit):
    rv.run_SciPyOp(fit)
    if fit.bound_error or not np.isfinite(fit.loglik):
        raise RuntimeError(fit.bound_error_msg or 'Nonfinite transit likelihood')
    fit.tra_jitt_use[0] = True
    rv.run_SciPyOp(fit)
    if fit.bound_error or not np.isfinite(fit.loglik):
        raise RuntimeError(fit.bound_error_msg or 'Nonfinite transit likelihood')
    print(f'Transit lnL = {fit.loglik:.4f}')
    for key in ('P', 't0', 'pl_a', 'pl_rad', 'i'):
        print(key, getattr(fit, key)[0])
    print('Optimizer diagnostics:', fit.scipy_fit_diagnostics)
    check_priors(fit)
    return fit

fit = optimize(fit)

## 4. Inspect and save the best fit

The figures show the fitted model and residuals. RV error bars include each
instrument's fitted jitter in quadrature; phase plots remove the other planet's
signal. For WASP-6, both the full time series and the folded transit are shown.
The `.ses` output can be reopened in the Exo-Striker GUI. Re-running this notebook
replaces files of the same name inside `OUTPUT`; change that directory to keep
separate analyses.

In [ ]:
plot_transit(fit, OUTPUT, TARGET)
save_session(fit, OUTPUT, TARGET)
plt.show()

## 5. Check the sampler parameter space

Inspect every free parameter and its bounds. Sampling uses the same physical
model and fixed assumptions as the final fit above. Starting outside the bounds
is an error; a value near a boundary deserves investigation. Do not silently
shrink priors around this fitted result to accelerate evidence calculations.

In [ ]:
check_priors(fit)

## 6. Optional nested sampling

Dynamic nested sampling uses `rwalk` proposals and multi-ellipsoid bounds.
`nlive = live_points_fact × number of free parameters`; `stop_crit=0.1` controls
the initial evidence tolerance. `ns_pfrac=1.0` allocates work for the posterior.
The iteration and likelihood-call caps are disabled by default. The settings
below are starting choices; convergence must be checked for each analysis.

The current library keeps equal-weight `samples` and `logl` aligned and retains
raw nested samples in `samples_ns` and raw likelihoods in `logl_ns`. The historical
weight-alignment warning in the ν Oph notebook no longer describes this version.
Keep `ns_save_sampler=True` to retain the arrays needed for diagnostics and export.
Scripts protect execution with `if __name__ == '__main__'` for multiprocessing;
use `NS_THREADS=1` initially in notebooks.

In [ ]:
configure_sampling(fit, threads=NS_THREADS)
print('Free parameters:', len(fit.par_for_mcmc))
print('Initial live points:', fit.live_points_fact * len(fit.par_for_mcmc))
print('Run nested sampling:', RUN_NESTED)
if RUN_NESTED:
    fit = rv.run_nestsamp(fit)
    if fit.bound_error:
        raise RuntimeError(fit.bound_error_msg)
else:
    print('Skipped. Set RUN_NESTED=True and rerun this cell to sample.')

## 7. Diagnostics, posterior exports, and corner plot

Raw nested-sampling rows carry unequal weights. The export utility normalizes
`logwt`, computes **ESS = 1 / sum(weight²)** and uses systematic resampling with
seed 42, applying the same row indices to parameter vectors and their `logL`.
It prints iteration/call counts, efficiency, evidence and evidence uncertainty.
Low ESS (the helper flags <200) is a warning; a larger ESS alone does not prove
convergence. Check repeat runs, posterior structure, integration settings where
relevant, and stopping criteria before interpreting evidence or uncertainties.

Outputs include an equal-weight posterior CSV, raw samples with weights and
likelihoods, median/68.3% interval summary CSV, and PNG/PDF corner plots. Column
indices and original labels identify parameters without assuming a packing order.
The maximum-likelihood vector comes from the raw aligned arrays. Evidence is
conditional on the chosen likelihood, fixed assumptions, and prior volume.
The updated session retains the sampler for subsequent inspection.

In [ ]:
if RUN_NESTED:
    posterior = export_posterior(fit, OUTPUT, TARGET)
    save_session(fit, OUTPUT, TARGET + '_NS')
    plt.show()
else:
    print('No posterior exported: nested sampling has not been run.')

## Attribution

**These examples are provided by Trifon Trifonov with the aid of CODEX; use with caution.** Their organization follows the
ν Oph workflow in Alan's notebooks. The observations are the files bundled with
Exo-Striker; this attribution identifies the provider of the examples and does
not assert new ownership of the observational data.